In [ ]:
from __future__ import annotations
from pathlib import Path

import pandas as pd
from rdkit import Chem

INPUT_PATH = Path("input_file.csv")
OUTPUT_PATH = Path("ic50_clean.csv")

# Колонки, которые оставляем в конечном файле
REQUIRED_COLS = [
    "Molecule ChEMBL ID",
    "Smiles",
    "Standard Relation",
    "Standard Value",
    "Standard Units",
]

#Единицы измерения, которые мы оставляем (другие единицы нельзя преобразовать в nM)
NANOMOLAR_TAGS = {"nm", "nM"}

Делаем проверку SMILES, если запись пустая или не является строчкой сразу откидываем. Если rdkit не может интерпретировать запись также выкидываем (Выдаёт значение bool)

In [ ]:
def valid_smiles(smiles):
    if not isinstance(smiles, str) or not smiles:
        return False
    try:
        return Chem.MolFromSmiles(smiles) is not None
    except Exception:
        return False

Поэтапно убирает колонки, убирает дубликаты, неподходящие удиницы измерения и не валидные SMILES

In [ ]:
def clean_ic50(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, sep=";", quotechar='"', low_memory=False)

    # 1. Проверяем, все ли нужные колонки присутствуют
    missing = set(REQUIRED_COLS) - set(df.columns)
    if missing:
        raise ValueError(f"Пропущена колонка: {', '.join(sorted(missing))}")

    df = df.loc[:, REQUIRED_COLS].copy()

    # 2. IC50 должны быть числами
    df["Standard Value"] = pd.to_numeric(df["Standard Value"], errors="coerce")
    df = df.dropna(subset=["Standard Value"])

    # 3. Оставляем только значения в нМ
    df["Standard Units"] = df["Standard Units"].str.replace(" ", "").str.lower()
    df = df[df["Standard Units"].isin({u.lower() for u in NANOMOLAR_TAGS})]

    # 4. Удаляем дубликаты по ID
    df = df.drop_duplicates(subset="Molecule ChEMBL ID", keep="first")

    # 5. Проверяем SMILES
    df = df[df["Smiles"].map(valid_smiles)]
    return df[['Molecule ChEMBL ID', 'Smiles', 'Standard Relation', 'Standard Value (nM)']]

Создаёт очищенный csv в тйо же папке

In [ ]:
cleaned = clean_ic50(INPUT_PATH)
cleaned.to_csv(OUTPUT_PATH, index=False)